# NHS England Maternal Care Pathways Master Pipeline
## Stage 3 - Merge combined MSDS with mapped HES to fill missingness

### Compiled by Ethan Phillips (Oxford)

### Last updated 10.11.2025

In [0]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql import SparkSession 
from pyspark.sql.functions import when, col, lit, count, max, last, first, min, avg, sum, collect_list, collect_set, countDistinct, to_date, datediff, array_contains, size, udf, format_number, regexp_replace, explode, array, array_union, desc_nulls_last
from pyspark.sql.types import IntegerType, StringType, NumericType, DateType, ArrayType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
from pyspark.ml.regression import GeneralizedLinearRegression as GLR

%run "/Workspace/Shared/Helper_Methods_EP"


In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
read_table_name_1 = "msds_all_agg_filtered_stage"
read_table_name_2 = "msds_hes_mapping_stage"
read_table_name_3 = "hes_aggregated_LD_spells_stage"

df_master = spark.read.table(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{read_table_name_1}")
msds_hes_map = spark.read.table(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{read_table_name_2}")
df_hes_apc = spark.read.table(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{read_table_name_3}")

In [0]:
df_master = df_master.join(msds_hes_map, on="UniqPregID", how="left")

In [0]:
df_master = df_master.withColumn(
    "birth_weight",
    col("birth_weight").try_cast("float")
).withColumn(
    "birth_weight", 
    when(col("birth_weight") > 7001, None)\
    .when(col("birth_weight") < 10, col("birth_weight")*1000)\
    .when(col("birth_weight") < 1000, None)\
    .otherwise(col("birth_weight"))
)

print(f"Missingness in birthweight prior to fill: {(df_master.filter(col("birth_weight").isNull()).count())/(df_master.count())}")

df_master = fill_nulls_from_hes(df_master, df_hes_apc, "birth_weight", "BIRWEIT")

df_master = df_master.withColumn(
    "birth_weight", 
    when(col("birth_weight") > 7001, None)\
    .when(col("birth_weight") < 10, col("birth_weight")*1000)\
    .when(col("birth_weight") < 1000, None)\
    .otherwise(col("birth_weight"))
)

print(f"Missingness in birthweight after fill: {(df_master.filter(col("birth_weight").isNull()).count())/(df_master.count())}")

In [0]:
df_master = df_master.withColumn(
    "pregnancy_outcome",
    when(~col("pregnancy_outcome").isin(1, 2, 3, 4), None)\
    .otherwise(col("pregnancy_outcome"))
)

print(f"Missingness in birth outcome prior to fill: {(df_master.filter(col("pregnancy_outcome").isNull()).count())/(df_master.count())}")

df_master = fill_nulls_from_hes(df_master, df_hes_apc, "pregnancy_outcome", "BIRSTAT")

df_master = df_master.withColumn(
    "pregnancy_outcome",
    when(~col("pregnancy_outcome").isin(1, 2, 3, 4), None)\
    .otherwise(col("pregnancy_outcome"))
)

print(f"Missingness in birth outcome after fill: {(df_master.filter(col("pregnancy_outcome").isNull()).count())/(df_master.count())}")


In [0]:
# gestation length in msds is coded in days
df_master = df_master.withColumn(
    "gest_at_booking", 
    when(col("gest_at_booking") < 50, (col("gest_at_booking")*7).cast("int"))\
    .otherwise(col("gest_at_booking").cast("int"))
).withColumn(
    "gest_at_booking",
    when(col("gest_at_booking") < 7*10, None)\
    .when(col("gest_at_booking") > 7*49, None)\
    .otherwise(col("gest_at_booking").cast("int"))
)

# GESTAT_1 in hes_apc is coded in weeks
df_hes_apc = df_hes_apc.withColumn(
    "hes_gestation_length_booking",
    when(col("ANAGEST") < 10, None)\
    .when(col("ANAGEST") > 49, None)\
    .otherwise((col("ANAGEST")*7).cast("int"))
)

print(f"Missingness in gestation length prior to fill: {(df_master.filter(col("gest_at_booking").isNull()).count())/(df_master.count())}")

df_master = fill_nulls_from_hes(df_master, df_hes_apc, "gest_at_booking", "hes_gestation_length_booking")

df_master = df_master.withColumn(
    "gest_at_booking",
    when(col("gest_at_booking") < 7*10, None)\
    .when(col("gest_at_booking") > 7*49, None)\
    .otherwise(col("gest_at_booking").cast("int"))
)

print(f"Missingness in gestation length after fill: {(df_master.filter(col("gest_at_booking").isNull()).count())/(df_master.count())}")


In [0]:
# gestation length in msds is coded in days
df_master = df_master.withColumn(
    "gestation_length_at_birth", 
    when(col("gestation_length_at_birth") < 50, (col("gestation_length_at_birth")*7).cast("int"))\
    .otherwise(col("gestation_length_at_birth").cast("int"))
).withColumn(
    "gestation_length_at_birth",
    when(col("gestation_length_at_birth") < 7*10, None)\
    .when(col("gestation_length_at_birth") > 7*49, None)\
    .otherwise(col("gestation_length_at_birth").cast("int"))
)

# GESTAT_1 in hes_apc is coded in weeks
df_hes_apc = df_hes_apc.withColumn(
    "hes_gestation_length",
    when(col("GESTAT") < 10, None)\
    .when(col("GESTAT") > 49, None)\
    .otherwise((col("GESTAT")*7).cast("int"))
)

print(f"Missingness in gestation length prior to fill: {(df_master.filter(col("gestation_length_at_birth").isNull()).count())/(df_master.count())}")

df_master = fill_nulls_from_hes(df_master, df_hes_apc, "gestation_length_at_birth", "hes_gestation_length")

df_master = df_master.withColumn(
    "gestation_length_at_birth",
    when(col("gestation_length_at_birth") < 7*10, None)\
    .when(col("gestation_length_at_birth") > 7*49, None)\
    .otherwise(col("gestation_length_at_birth").cast("int"))
)

print(f"Missingness in gestation length after fill: {(df_master.filter(col("gestation_length_at_birth").isNull()).count())/(df_master.count())}")


In [0]:
df_master = df_master.withColumn(
    "delivery_method", when(col("delivery_method") > 9, None).otherwise(col("delivery_method"))
)

print(f"Missingness in delivery method prior to fill: {(df_master.filter(col("delivery_method").isNull()).count())/(df_master.count())}")
()
df_master = fill_nulls_from_hes(df_master, df_hes_apc, "delivery_method", "DELMETH")

df_master = df_master.withColumn(
    "delivery_method", when(col("delivery_method") > 9, None).otherwise(col("delivery_method"))
)

print(f"Missingness in delivery method after fill: {(df_master.filter(col("delivery_method").isNull()).count())/(df_master.count())}")

In [0]:
print(f"Missingness in L&D hospital admission date prior to fill: {(df_master.filter(col("ld_hosp_start_date").isNull()).count())/(df_master.count())}")

df_master = fill_nulls_from_hes(df_master, df_hes_apc, "ld_hosp_start_date", "ADMIDATE")

print(f"Missingness in L&D hospital admission date after fill: {(df_master.filter(col("ld_hosp_start_date").isNull()).count())/(df_master.count())}")

In [0]:
print(f"Missingness in L&D hospital discharge date prior to fill: {(df_master.filter(col("ld_disch_date").isNull()).count())/(df_master.count())}")

df_master = fill_nulls_from_hes(df_master, df_hes_apc, "ld_disch_date", "DISDATE")

print(f"Missingness in L&D hospital discharge date after fill: {(df_master.filter(col("ld_disch_date").isNull()).count())/(df_master.count())}")

In [0]:
print(f"Missingness in L&D site id prior to fill: {(df_master.filter(col("ld_site_id").isNull()).count())/(df_master.count())}")

df_master = fill_nulls_from_hes(df_master, df_hes_apc, "ld_site_id", "SITETRET")

print(f"Missingness in L&D site id after fill: {(df_master.filter(col("ld_site_id").isNull()).count())/(df_master.count())}")


In [0]:
print(f"Missingness in number of births prior to fill: {(df_master.filter(col("num_births").isNull()).count())/(df_master.count())}")

df_master = fill_nulls_from_hes(df_master, df_hes_apc, "num_births", "NUMBABY")

print(f"Missingness in number of births after fill: {(df_master.filter(col("num_births").isNull()).count())/(df_master.count())}")


In [0]:
print(f"Missingness in birth year prior to birthday calculation: {(df_master.filter(col("birth_year").isNull()).count())/(df_master.count())}")

df_master = df_master.withColumn(
    "birth_date",
    when(((col("antenatal_appt_date").isNotNull()) & (col("gest_at_booking").isNotNull()) & (col("gestation_length_at_birth").isNotNull())), col("antenatal_appt_date") - col("gest_at_booking").cast("int") + col("gestation_length_at_birth").cast("int")).otherwise(None)
).withColumns({
    "birth_year": when(col("birth_date").isNotNull(), F.year(col("birth_date"))).otherwise(col("birth_year")),
    "birth_month": when(col("birth_date").isNotNull(), F.month(col("birth_date"))).otherwise(col("birth_month"))
})

print(f"Missingness in birth year after birthday calculation: {(df_master.filter(col("birth_year").isNull()).count())/(df_master.count())}")

In [0]:
print(f"Missingness in birth year prior to fill: {(df_master.filter(col("birth_year").isNull()).count())/(df_master.count())}")

df_master = fill_nulls_from_hes(df_master, df_hes_apc, "birth_year", "BIRYEAR")

print(f"Missingness in birth year after fill: {(df_master.filter(col("birth_year").isNull()).count())/(df_master.count())}")

In [0]:
write_table_name = "msds_joined_filtered_filled_stage"

df_master.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{write_table_name}")